# SUMO Scenario Preparation

This notebook prepares the SUMO scenario validation layer for the AI-driven decision support system.

SUMO is used as a focused decision-support validation engine to test whether recommended transport interventions may improve operational conditions under selected route or corridor scenarios.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
# Load decision engine output

PROJECT_ROOT = Path.cwd().parent

decision_path = PROJECT_ROOT / "data" / "processed" / "decision_engine_output.csv"

decision_df = pd.read_csv(decision_path)

print("Decision engine dataset shape:", decision_df.shape)

decision_df.head()

Decision engine dataset shape: (2000, 7)


,route_id,trip_id,actual_arrival_delay_min,delay_risk,recommended_action,weather_condition,traffic_congestion_index
0,Route_15,T00000,3,Low,No operational action required,Storm,81
1,Route_12,T00001,9,Medium,Monitor route conditions,Rain,53
2,Route_16,T00002,0,Low,No operational action required,Clear,67
3,Route_19,T00003,10,Medium,Monitor route conditions,Clear,84
4,Route_8,T00004,14,Medium,Monitor route conditions,Snow,46


In [4]:
# Inspect recommendation distribution

decision_df["recommended_action"].value_counts()

Monitor route conditions          636
Adjust service headway            627
No operational action required    501
Deploy standby bus                236
Name: recommended_action, dtype: int64

In [5]:
# Select candidate records for SUMO validation

sumo_candidates = decision_df[
    decision_df["recommended_action"].isin([
        "Adjust service headway",
        "Deploy standby bus"
    ])
].copy()

print("SUMO candidate records:", sumo_candidates.shape)

sumo_candidates.head()

SUMO candidate records: (863, 7)


,route_id,trip_id,actual_arrival_delay_min,delay_risk,recommended_action,weather_condition,traffic_congestion_index
6,Route_5,T00006,28,Severe,Deploy standby bus,Storm,22
7,Route_16,T00007,23,High,Adjust service headway,Clear,85
8,Route_6,T00008,19,High,Adjust service headway,Clear,64
11,Route_12,T00011,21,High,Adjust service headway,Cloudy,91
13,Route_9,T00013,22,High,Adjust service headway,Storm,62


In [6]:
# Identify route candidates with frequent high-risk recommendations

route_priority = (
    sumo_candidates
    .groupby("route_id")
    .agg(
        total_cases=("trip_id", "count"),
        avg_delay_min=("actual_arrival_delay_min", "mean"),
        max_delay_min=("actual_arrival_delay_min", "max"),
        avg_congestion=("traffic_congestion_index", "mean")
    )
    .sort_values(
        by=["total_cases", "avg_delay_min"],
        ascending=False
    )
    .reset_index()
)

route_priority.head(10)

,route_id,total_cases,avg_delay_min,max_delay_min,avg_congestion
0,Route_12,53,21.471698,28,53.018868
1,Route_3,51,22.666667,29,51.784314
2,Route_18,50,23.080000,29,53.780000
3,Route_8,48,22.583333,29,50.625000
4,Route_5,48,22.395833,29,48.375000
5,Route_4,48,21.208333,28,48.666667
6,Route_15,45,23.355556,29,48.555556
7,Route_14,45,22.400000,29,51.866667
8,Route_2,45,21.844444,29,41.355556
9,Route_20,44,23.022727,29,52.886364


# Selected SUMO Validation Corridor

Route_12 was selected as the primary SUMO validation candidate due to its high frequency of operational intervention recommendations and elevated average delay conditions.

The SUMO validation stage will simulate whether operational actions such as service headway adjustment or standby bus deployment may reduce congestion and delay propagation under selected disruption conditions.

In [7]:
# Select Route_12 for SUMO scenario preparation

selected_route = "Route_12"

sumo_route_df = route_priority[
    route_priority["route_id"] == selected_route
]

sumo_route_df

,route_id,total_cases,avg_delay_min,max_delay_min,avg_congestion
0,Route_12,53,21.471698,28,53.018868


In [8]:
# Extract Route_12 operational records

route_12_cases = sumo_candidates[
    sumo_candidates["route_id"] == selected_route
].copy()

print("Selected SUMO records:", route_12_cases.shape)

route_12_cases.head()

Selected SUMO records: (53, 7)


,route_id,trip_id,actual_arrival_delay_min,delay_risk,recommended_action,weather_condition,traffic_congestion_index
11,Route_12,T00011,21,High,Adjust service headway,Cloudy,91
46,Route_12,T00046,22,High,Adjust service headway,Cloudy,56
57,Route_12,T00057,22,High,Adjust service headway,Fog,78
70,Route_12,T00070,21,High,Adjust service headway,Storm,72
149,Route_12,T00149,22,High,Adjust service headway,Snow,14


In [9]:
# Scenario summary statistics

scenario_summary = route_12_cases.agg({
    "actual_arrival_delay_min": ["mean", "max"],
    "traffic_congestion_index": ["mean", "max"]
})

scenario_summary

,actual_arrival_delay_min,traffic_congestion_index
mean,21.471698,53.018868
max,28.000000,94.000000


# SUMO Validation Scenarios

Three operational scenarios are defined for SUMO-based validation.

## 1. Baseline Scenario
Represents normal public transport operations under standard traffic conditions without intervention.

## 2. Disruption Scenario
Represents elevated congestion and high-delay conditions observed on Route_12 using the integrated GTFS and weather dataset.

## 3. Intervention Scenario
Represents operational intervention strategies recommended by the AI decision engine, including:
- Service headway adjustment
- Standby bus deployment

The objective is to evaluate whether operational interventions may reduce delay propagation and congestion impact under simulated disruption conditions.

In [10]:
# Create SUMO scenario definition table

sumo_scenarios = pd.DataFrame({
    "scenario_name": [
        "Baseline Operations",
        "Disruption Scenario",
        "Intervention Scenario"
    ],

    "description": [
        "Normal transport operations without disruption",
        "High congestion and severe delay conditions on Route_12",
        "Operational intervention using AI recommendations"
    ],

    "selected_route": [
        selected_route,
        selected_route,
        selected_route
    ],

    "avg_delay_min": [
        5,
        round(route_12_cases["actual_arrival_delay_min"].mean(), 2),
        12
    ],

    "traffic_congestion_index": [
        35,
        round(route_12_cases["traffic_congestion_index"].mean(), 2),
        40
    ],

    "recommended_action": [
        "None",
        "Monitor conditions",
        "Adjust service headway"
    ]
})

sumo_scenarios

,scenario_name,description,selected_route,avg_delay_min,traffic_congestion_index,recommended_action
0,Baseline Operations,Normal transport operations without disruption,Route_12,5.00,35.00,None
1,Disruption Scenario,High congestion and severe delay conditions on...,Route_12,21.47,53.02,Monitor conditions
2,Intervention Scenario,Operational intervention using AI recommendations,Route_12,12.00,40.00,Adjust service headway


In [11]:
# Save SUMO scenario definitions

output_path = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "sumo_scenarios.csv"
)

sumo_scenarios.to_csv(output_path, index=False)

print("SUMO scenario definitions saved.")
print(output_path)

SUMO scenario definitions saved.
d:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\sumo_scenarios.csv



The SUMO validation stage evaluates whether AI-recommended interventions may improve operational transport conditions under selected disruption scenarios.

The following validation metrics are defined:

- Average vehicle delay
- Maximum corridor delay
- Traffic congestion index
- Queue propagation impact
- Service reliability improvement
- Operational intervention effectiveness

The simulation compares baseline, disruption, and intervention scenarios using Route_12 as the selected operational corridor.

In [13]:
# Define SUMO validation KPIs

sumo_kpis = pd.DataFrame({
    "metric": [
        "Average Delay",
        "Maximum Delay",
        "Congestion Index",
        "Queue Length",
        "Service Reliability",
        "Intervention Effectiveness"
    ],

    "description": [
        "Average vehicle delay across the corridor",
        "Maximum observed delay during simulation",
        "Traffic congestion severity indicator",
        "Estimated traffic queue propagation",
        "Transport schedule consistency",
        "Measured improvement after intervention"
    ],

    "baseline_target": [
        "< 10 min",
        "< 15 min",
        "< 40",
        "Low",
        "Stable",
        "N/A"
    ],

    "intervention_goal": [
        "Reduce delay",
        "Reduce peak disruption",
        "Reduce congestion",
        "Reduce queue buildup",
        "Improve consistency",
        "Improve corridor performance"
    ]
})

sumo_kpis

,metric,description,baseline_target,intervention_goal
0,Average Delay,Average vehicle delay across the corridor,< 10 min,Reduce delay
1,Maximum Delay,Maximum observed delay during simulation,< 15 min,Reduce peak disruption
2,Congestion Index,Traffic congestion severity indicator,< 40,Reduce congestion
3,Queue Length,Estimated traffic queue propagation,Low,Reduce queue buildup
4,Service Reliability,Transport schedule consistency,Stable,Improve consistency
5,Intervention Effectiveness,Measured improvement after intervention,N/A,Improve corridor performance


In [14]:
# Save SUMO KPI definitions

kpi_output_path = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "sumo_kpis.csv"
)

sumo_kpis.to_csv(kpi_output_path, index=False)

print("SUMO KPI definitions saved.")
print(kpi_output_path)

SUMO KPI definitions saved.
d:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\sumo_kpis.csv


# SUMO Operational Assumptions

The SUMO validation stage uses a simplified operational corridor abstraction rather than a full Auckland-wide transport network simulation.

The following assumptions are applied:

- Route_12 represents the selected operational corridor.
- Vehicle interactions are modeled under simplified traffic flow conditions.
- Weather disruption effects are represented through congestion escalation and delay propagation.
- AI interventions modify operational behavior through:
  - service headway adjustment
  - standby bus deployment
- Simulation outputs are evaluated comparatively rather than as exact real-world predictions.

This approach supports computational feasibility while preserving decision-support validity for the capstone scope.

In [15]:
# Define simplified SUMO corridor structure

sumo_corridor = pd.DataFrame({
    "corridor_segment": [
        "Segment_A",
        "Segment_B",
        "Segment_C",
        "Segment_D"
    ],

    "description": [
        "Urban entry corridor",
        "High congestion mid-route section",
        "Commercial district zone",
        "Terminal approach corridor"
    ],

    "expected_condition": [
        "Moderate flow",
        "Heavy congestion",
        "Variable traffic density",
        "Delay propagation risk"
    ]
})

sumo_corridor

,corridor_segment,description,expected_condition
0,Segment_A,Urban entry corridor,Moderate flow
1,Segment_B,High congestion mid-route section,Heavy congestion
2,Segment_C,Commercial district zone,Variable traffic density
3,Segment_D,Terminal approach corridor,Delay propagation risk


In [16]:
# Define intervention logic for SUMO validation

intervention_logic = pd.DataFrame({
    "delay_risk": [
        "Low",
        "Medium",
        "High",
        "Severe"
    ],

    "recommended_action": [
        "No operational action required",
        "Monitor route conditions",
        "Adjust service headway",
        "Deploy standby bus"
    ],

    "expected_simulation_effect": [
        "Stable corridor performance",
        "Early congestion detection",
        "Reduced delay propagation",
        "Rapid recovery from disruption"
    ]
})

intervention_logic

,delay_risk,recommended_action,expected_simulation_effect
0,Low,No operational action required,Stable corridor performance
1,Medium,Monitor route conditions,Early congestion detection
2,High,Adjust service headway,Reduced delay propagation
3,Severe,Deploy standby bus,Rapid recovery from disruption


In [17]:
# Save intervention logic definitions

logic_output_path = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "intervention_logic.csv"
)

intervention_logic.to_csv(logic_output_path, index=False)

print("Intervention logic definitions saved.")
print(logic_output_path)

Intervention logic definitions saved.
d:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\intervention_logic.csv
